In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, Trace
from pydantic import BaseModel
from agents.mcp import MCPServerStdio
import os

load_dotenv()

True

In [ ]:
agent = Agent(
    name="Jokester",
    instructions="You tell nerdy jokes and turn any question into a joke",
    tools=[],    
    model="gpt-4o-mini",
)


In [ ]:
result = await Runner.run(agent, "What is the capital of France?")
print(result.final_output)

In [2]:
fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    fetch_tools = await server.list_tools()

fetch_tools

[Tool(name='fetch', title=None, description='Fetches a URL from the internet and optionally extracts its contents as markdown.\n\nAlthough originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.', inputSchema={'description': 'Parameters for fetching a URL.', 'properties': {'url': {'description': 'URL to fetch', 'format': 'uri', 'minLength': 1, 'title': 'Url', 'type': 'string'}, 'max_length': {'default': 5000, 'description': 'Maximum number of characters to return.', 'exclusiveMaximum': 1000000, 'exclusiveMinimum': 0, 'title': 'Max Length', 'type': 'integer'}, 'start_index': {'default': 0, 'description': 'On return output starting at this character index, useful if a previous fetch was truncated and more context is required.', 'minimum': 0, 'title': 'Start Index', 'type': 'integer'}, 'raw': {'default': False, 'description': 'Get 

In [3]:
import os
from dotenv import load_dotenv
from agents.mcp import MCPServerStdio

load_dotenv(override=True)

mssql_server = MCPServerStdio(
    params={
        "command": "mssql-mcp-server",   # or "npx", "args": ["-y", "mssql-mcp-server"]
        "env": {
            "MSSQL_CONNECTION_STRING": os.getenv("MSSQL_CONNECTION_STRING"),
            # "MSSQL_HOST": os.getenv("MSSQL_HOST"),
            # "MSSQL_PORT": os.getenv("MSSQL_PORT", "1433"),
            # "MSSQL_DATABASE": os.getenv("MSSQL_DATABASE"),
            # "MSSQL_USER": os.getenv("MSSQL_USER"),
            # "MSSQL_PASSWORD": os.getenv("MSSQL_PASSWORD"),
            # "MSSQL_ENCRYPT": os.getenv("MSSQL_ENCRYPT", "false"),
            # "MSSQL_TRUST_SERVER_CERTIFICATE": os.getenv("MSSQL_TRUST_SERVER_CERTIFICATE", "true"),
        },
    },
    client_session_timeout_seconds=60,
)

async with mssql_server:
    tools = await mssql_server.list_tools()
    print([t.name for t in tools])


['list_connections', 'test_connection', 'list_databases', 'list_tables', 'describe_table', 'sample_data', 'execute_query', 'get_relationships', 'list_stored_procedures', 'describe_stored_procedure', 'list_views', 'describe_view', 'list_triggers', 'describe_trigger', 'list_functions', 'list_indexes', 'analyze_table_stats', 'find_missing_indexes', 'analyze_index_usage', 'analyze_database_size', 'list_constraints', 'analyze_check_constraints', 'list_user_defined_types', 'find_computed_columns', 'list_default_constraints', 'analyze_data_distribution', 'find_lookup_tables', 'analyze_null_patterns', 'detect_audit_columns', 'get_stored_procedure_definition', 'get_multiple_stored_procedure_definitions', 'get_all_stored_procedure_definitions', 'search_stored_procedures_by_content']


In [4]:
load_dotenv(override=True)

async with mssql_server:
    print(await mssql_server.call_tool("test_connection", {}))
    #print(await mssql_server.call_tool("list_tables", {}))


meta=None content=[TextContent(type='text', text='[\n  {\n    "version": "Microsoft SQL Server 2019 (RTM) - 15.0.2000.5 (X64) \\n\\tSep 24 2019 13:48:23 \\n\\tCopyright (C) 2019 Microsoft Corporation\\n\\tExpress Edition (64-bit) on Windows 10 Home 10.0 <X64> (Build 26200: ) (Hypervisor)\\n",\n    "server_name": "LAPTOP-CPI3JAKQ\\\\SQLEXPRESS",\n    "database_name": "Certainty_Durham2"\n  }\n]', annotations=None, meta=None)] structuredContent=None isError=False


In [ ]:
# async with mssql_server:
#     out = await mssql_server.call_tool("list_tables", {})
#     print(out)


meta=None content=[TextContent(type='text', text='Error listing tables: No connection string provided and no default connection string configured', annotations=None, meta=None)] structuredContent=None isError=True


In [ ]:
# async with mssql_server:
#     agent = Agent(
#         name="SQL Agent",
#         instructions="Use database tools to answer questions. Prefer SELECT statements unless asked to modify data.",
#         model="gpt-4o-mini",
#         mcp_servers=[mssql_server],
#     )

#     result = await Runner.run(agent, "List available tables.")
#     print(result.final_output)



In [ ]:
async with mssql_server:
    agent = Agent(
        name="SQL Analyst",
        model="gpt-4o-mini",
        instructions=(
            "You are a SQL Server analyst.\n"
            "Workflow:\n"
            "1) Use list_tables to find candidates.\n"
            "2) Use describe_table before writing SQL.\n"
            "3) Use execute_query for final SQL.\n"
            "4) Return concise results and include the SQL used."
        ),
        mcp_servers=[mssql_server],
    )

    result = await Runner.run(
        agent,
        "Find the top 10 most recently created completed checklists with site name and created date."
    )
    print(result.final_output)


UserError: Server not initialized. Make sure you call `connect()` first.

[non-fatal] Tracing client error 400: {
  "error": {
    "message": "Invalid type for 'data[1].span_data.result': expected an array of strings, but got null instead.",
    "type": "invalid_request_error",
    "param": "data[1].span_data.result",
    "code": "invalid_type"
  }
}


In [7]:
load_dotenv(override=True)

async with mssql_server:
    tools = await mssql_server.list_tools()
    print([t.name for t in tools])
    result = await Runner.run(agent, "Use one database tool call to run SELECT TOP 5 name FROM sys.tables")
    print(result.final_output)


['list_connections', 'test_connection', 'list_databases', 'list_tables', 'describe_table', 'sample_data', 'execute_query', 'get_relationships', 'list_stored_procedures', 'describe_stored_procedure', 'list_views', 'describe_view', 'list_triggers', 'describe_trigger', 'list_functions', 'list_indexes', 'analyze_table_stats', 'find_missing_indexes', 'analyze_index_usage', 'analyze_database_size', 'list_constraints', 'analyze_check_constraints', 'list_user_defined_types', 'find_computed_columns', 'list_default_constraints', 'analyze_data_distribution', 'find_lookup_tables', 'analyze_null_patterns', 'detect_audit_columns', 'get_stored_procedure_definition', 'get_multiple_stored_procedure_definitions', 'get_all_stored_procedure_definitions', 'search_stored_procedures_by_content']
Here are the top 5 table names from the database:

1. **ChecklistStructure_ConditionalQuestion_SiteTrigger**
2. **ChecklistTemplate_QuestionSet**
3. **ChecklistWorkflowStageStatus**
4. **ChecklistTemplate_Section**
5

In [ ]:
import pyodbc
import subprocess

host = subprocess.check_output("ip route | awk '/default/ {print $3}'", shell=True).decode().strip()

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={host},1433;"
    "DATABASE=Certainty_Durham2;"
    "UID=analytics_user;"
    "PWD=UseAStrongPasswordHere!;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

#command:  ip route | grep default

# conn = pyodbc.connect(
#     "DRIVER={ODBC Driver 18 for SQL Server};"
#     "SERVER=172.27.112.1,1433;"
#     "DATABASE=Certainty_Durham2;"
#     "UID=analytics_user;"
#     "PWD=UseAStrongPasswordHere!;"
#     "Encrypt=yes;"
#     "TrustServerCertificate=yes;"
# )
cur = conn.cursor()
cur.execute("SELECT DB_NAME(), SUSER_SNAME(), @@VERSION")
print(cur.fetchone())


In [ ]:
import pymssql

conn = pymssql.connect(
    server=host,  # or f"{host}:1433"
    port=1433,
    user="analytics_user",
    password="UseAStrongPasswordHere!",
    database="Certainty_Durham2",
    # timeout=30,  # optional
    # login_timeout=10,  # optional
)

try:
    with conn.cursor() as cur:
        cur.execute("SELECT @@VERSION AS version")
        row = cur.fetchone()
        print(row[0])
finally:
    conn.close()
